# Sync Results — API → match_external_links → DB

Update match results using `match_external_links` to map provider IDs to internal UUIDs.

**Flow:**
1. Find pending matches via `match_external_links` JOIN
2. Fetch from API, match by external_id
3. Compare — show diff
4. UPDATE only `status` + `scores`
5. Recalculate prediction scores for finished matches

---
## 1. Setup

In [ ]:
import json
import os
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")
sys.path.insert(0, str(Path("..").resolve()))

API_KEY = os.environ.get("FOOTBALL_DATA_API_TOKEN")
BASE_URL = os.environ.get("FOOTBALL_DATA_API_BASE_URL", "https://api.football-data.org/v4")
COMPETITION = os.environ.get("FOOTBALL_DATA_COMPETITION", "WC")
HEADERS = {"X-Auth-Token": API_KEY}

PROVIDER = "football-data.org"


def api_get(endpoint: str) -> dict:
    """Call football-data.org API."""
    resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


def save_raw(data: dict | list, filename: str) -> Path:
    """Save raw JSON to data/raw/."""
    path = Path("../data/raw") / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved: {path} ({path.stat().st_size / 1024:.1f} KB)")
    return path


print(f"API Key: {'OK' if API_KEY else 'MISSING'}")
print(f"Competition: {COMPETITION}")
print(f"Provider: {PROVIDER}")

---
## 2. Async DB setup

In [ ]:
import asyncio

import nest_asyncio
from sqlalchemy import text as sa_text

from pipelines.common.db import get_engine, get_session
from pipelines.football_data.transform import STATUS_MAP

nest_asyncio.apply()

print("DB engine ready.")

---
## 3. Find pending matches

Matches where `match_date + 2h < now` and `status IN ('scheduled', 'live')`, joined via `match_external_links`.

In [ ]:
async def get_pending():
    engine = get_engine()
    async with engine.connect() as conn:
        result = await conn.execute(
            sa_text("""
            SELECT
                m.id,
                m.match_number,
                mel.external_id,
                m.stage::text,
                ht.code AS home,
                at.code AS away,
                m.match_date,
                m.status::text,
                m.home_score,
                m.away_score
            FROM matches m
            JOIN match_external_links mel ON mel.match_id = m.id AND mel.provider = :provider
            LEFT JOIN teams ht ON m.home_team_id = ht.id
            LEFT JOIN teams at ON m.away_team_id = at.id
            WHERE m.status::text IN ('scheduled', 'live')
              AND m.match_date + interval '2 hours' < now()
            ORDER BY m.match_number
        """),
            {"provider": PROVIDER},
        )
        rows = result.fetchall()
        columns = result.keys()
    await engine.dispose()
    return pd.DataFrame(rows, columns=columns)


df_pending = asyncio.get_event_loop().run_until_complete(get_pending())

if len(df_pending) == 0:
    print("No pending matches. All up to date!")
else:
    print(f"Pending matches ({len(df_pending)}):")
    print(df_pending.to_string(index=False))

---
## 4. Fetch from API

Only run if there are pending matches above.

In [ ]:
raw_matches = api_get(f"competitions/{COMPETITION}/matches")
save_raw(raw_matches, "matches.json")

# Build lookup by API match ID (as string to match external_id)
api_by_id = {str(m["id"]): m for m in raw_matches["matches"]}

print(f"Matches from API: {raw_matches['resultSet']['count']}")
status_counts = Counter(m["status"] for m in raw_matches["matches"])
for status, count in status_counts.most_common():
    print(f"  {status}: {count}")

---
## 5. Compare — show what changed

In [ ]:
changes = []

for _, row in df_pending.iterrows():
    ext_id = row["external_id"]
    api_match = api_by_id.get(ext_id)
    if not api_match:
        print(f"  WARNING: external_id {ext_id} not found in API")
        continue

    new_status = STATUS_MAP.get(api_match["status"], "scheduled")
    ft = api_match.get("score", {}).get("fullTime", {})
    new_home = ft.get("home") if new_status == "finished" else None
    new_away = ft.get("away") if new_status == "finished" else None

    if new_status == row["status"] and new_home == row["home_score"] and new_away == row["away_score"]:
        continue

    old_score = f"{row['home_score']}-{row['away_score']}" if pd.notna(row["home_score"]) else "null"
    new_score = f"{new_home}-{new_away}" if new_home is not None else "null"

    changes.append(
        {
            "match_id": row["id"],
            "external_id": ext_id,
            "home": row["home"],
            "away": row["away"],
            "old_status": row["status"],
            "new_status": new_status,
            "home_score": new_home,
            "away_score": new_away,
        }
    )
    print(f"  {ext_id}: {row['home']} vs {row['away']}  {row['status']} → {new_status}  {old_score} → {new_score}")

print(f"\nTotal changes: {len(changes)}")

---
## 6. Apply — UPDATE matches + recalculate predictions

**Check the diff above before running.**

In [ ]:
EXACT_SCORE_POINTS = 3
OUTCOME_POINTS = 1


def calc_points(pred_h, pred_a, real_h, real_a):
    """Returns (exact, outcome)."""
    if pred_h == real_h and pred_a == real_a:
        return EXACT_SCORE_POINTS, 0
    pred_out = (pred_h > pred_a) - (pred_h < pred_a)
    real_out = (real_h > real_a) - (real_h < real_a)
    if pred_out == real_out:
        return 0, OUTCOME_POINTS
    return 0, 0


async def apply_changes(changes):
    updated = 0
    scored = 0

    async with get_session() as session:
        for c in changes:
            await session.execute(
                sa_text("""
                    UPDATE matches
                    SET status = CAST(:status AS match_status),
                        home_score = :home_score,
                        away_score = :away_score
                    WHERE id = :match_id
                """),
                {
                    "match_id": str(c["match_id"]),
                    "status": c["new_status"],
                    "home_score": c["home_score"],
                    "away_score": c["away_score"],
                },
            )
            updated += 1

            if c["new_status"] == "finished" and c["home_score"] is not None:
                result = await session.execute(
                    sa_text("SELECT p.id, p.home_score, p.away_score FROM predictions p WHERE p.match_id = :mid"),
                    {"mid": str(c["match_id"])},
                )
                preds = result.fetchall()

                for p in preds:
                    exact, outcome = calc_points(p.home_score, p.away_score, c["home_score"], c["away_score"])
                    await session.execute(
                        sa_text("""
                            INSERT INTO prediction_scores
                                (prediction_id, exact_score_points, outcome_points, group_position_points, total_points)
                            VALUES (:pid, :exact, :outcome, 0, :exact + :outcome)
                            ON CONFLICT (prediction_id) DO UPDATE SET
                                exact_score_points = EXCLUDED.exact_score_points,
                                outcome_points = EXCLUDED.outcome_points,
                                total_points = EXCLUDED.exact_score_points + EXCLUDED.outcome_points
                                             + prediction_scores.group_position_points,
                                calculated_at = now()
                        """),
                        {"pid": str(p.id), "exact": exact, "outcome": outcome},
                    )
                    scored += 1

                print(
                    f"  {c['home']} vs {c['away']}: "
                    f"{c['home_score']}-{c['away_score']} "
                    f"- scored {len(preds)} prediction(s)"
                )

    return updated, scored


updated, scored = asyncio.get_event_loop().run_until_complete(apply_changes(changes))
print(f"\nUpdated {updated} match(es), scored {scored} prediction(s).")

---
## 7. Verify

In [ ]:
async def verify():
    engine = get_engine()
    async with engine.connect() as conn:
        result = await conn.execute(
            sa_text("""
            SELECT m.match_number, mel.external_id, m.stage::text,
                   ht.code AS home, m.home_score, m.away_score,
                   at.code AS away, m.status::text
            FROM matches m
            LEFT JOIN match_external_links mel ON mel.match_id = m.id AND mel.provider = :provider
            LEFT JOIN teams ht ON m.home_team_id = ht.id
            LEFT JOIN teams at ON m.away_team_id = at.id
            WHERE m.status::text = 'finished'
            ORDER BY m.match_number
        """),
            {"provider": PROVIDER},
        )
        finished = result.fetchall()
        cols = result.keys()

        result2 = await conn.execute(
            sa_text("SELECT status::text, COUNT(*) as total FROM matches GROUP BY status ORDER BY total DESC")
        )
        summary = result2.fetchall()
    await engine.dispose()
    return pd.DataFrame(finished, columns=cols), summary


df_finished, summary = asyncio.get_event_loop().run_until_complete(verify())

if len(df_finished) > 0:
    print(f"Finished matches ({len(df_finished)}):")
    print(df_finished.to_string(index=False))
else:
    print("No finished matches yet.")

print("\nSummary:")
for row in summary:
    print(f"  {row[0]}: {row[1]}")